# Relatório operacional de pré-decolagem

Notebook didático da issue #4. Ele executa cenários determinísticos do repositório e integra leitura, validação, energia e decisão. As faixas usadas são hipóteses didáticas; não representam parâmetros certificados de uma nave real.

## Preparação

O notebook funciona quando aberto na raiz do repositório ou dentro da pasta `notebooks/`. Não requer API key nem acesso à internet.

In [ ]:
from pathlib import Path
import sys

CANDIDATOS = (Path.cwd(), Path.cwd().parent)
RAIZ = next(p for p in CANDIDATOS if (p / 'src').is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.missao import LIMITES_PADRAO, executar_cenario

print(f'Repositório: {RAIZ}')
print(f'Limites didáticos: {LIMITES_PADRAO}')

## Função de apresentação temporária

A orquestração devolve dados estruturados. Enquanto `src/apresentacao.py` (issue #10) não estiver disponível, esta célula apenas exibe os campos de forma mínima para demonstrar a integração; ela não substitui a função de apresentação do colega.

In [ ]:
def exibir_cenario(resultado):
    print(f"Origem: {resultado['origem']}")
    print(f"Decisão: {resultado['decisao']}")
    if resultado['energia'] is not None:
        energia = resultado['energia']
        print(
            'Energia: inicial={:.2f} kWh | perdas={:.2f} kWh | '
            'útil={:.2f} kWh | saldo={:.2f} kWh | autonomia={} h'.format(
                energia['energia_inicial_kwh'],
                energia['perdas_kwh'],
                energia['energia_util_kwh'],
                energia['saldo_kwh'],
                energia['autonomia_h'],
            )
        )
    for motivo in resultado['motivos']:
        print(f'- {motivo}')

## Cenário nominal

O cenário nominal deve liberar a decolagem.

In [ ]:
nominal = executar_cenario(RAIZ / 'dados' / 'nominal.json', LIMITES_PADRAO)
exibir_cenario(nominal)

## Falha operacional

A propulsão em falha é um dado válido, mas a decisão deve abortar e informar o motivo.

In [ ]:
falha_modulo = executar_cenario(RAIZ / 'dados' / 'falha_modulo.json', LIMITES_PADRAO)
exibir_cenario(falha_modulo)

## Falha energética

A telemetria é válida, porém o saldo depois de perdas e consumo é insuficiente.

In [ ]:
falha_energia = executar_cenario(RAIZ / 'dados' / 'falha_energia.json', LIMITES_PADRAO)
exibir_cenario(falha_energia)

## Próxima integração

Quando a issue #10 entregar `src/apresentacao.py`, substituir `exibir_cenario` pela função oficial de apresentação e executar novamente o notebook do início ao fim. A geração por IA é opcional: os JSONs fixos mantêm esta demonstração reproduzível.